In [ ]:
import os
import zipfile
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.model_selection import KFold
import trimesh
import kagglehub

# Lista klas ModelNet10 (indeksy 0-9)
classes = ['bathtub', 'bed', 'chair', 'desk', 'dresser', 'monitor', 'night_stand', 'sofa', 'table', 'toilet']
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

# 2. Pobieranie danych via kagglehub
def download_modelnet10():
    """Pobiera dataset ModelNet10 via kagglehub."""
    path = kagglehub.dataset_download("balraj98/modelnet10-princeton-3d-object-dataset")
    print("Path to dataset files:", path)

    # Zakładamy, że path wskazuje na root datasetu z ModelNet10/
    data_dir = os.path.join(path, 'ModelNet10')
    if not os.path.exists(data_dir):
        data_dir = path
        print(f"Brak podkatalogu ModelNet10, używam bezpośredniego path: {data_dir}")

    # Sprawdź strukturę: oczekujemy katalogów klas z podkatalogami train/ i test/
    class_dir_example = os.path.join(data_dir, classes[0])  # np. bathtub
    train_dir_example = os.path.join(class_dir_example, 'train')
    test_dir_example = os.path.join(class_dir_example, 'test')
    if os.path.exists(train_dir_example) and os.path.exists(test_dir_example):
        print("Struktura potwierdzona: ModelNet10/<class_name>/train/ i test/")
    else:
        print("Błąd: Brak katalogów train/ lub test/ w", class_dir_example)
        print("Zawartość katalogu datasetu:")
        !ls -la {data_dir}
        print("\nZawartość przykładowego katalogu klasy (bathtub):")
        !ls -la {class_dir_example}
        raise FileNotFoundError("Nie znaleziono oczekiwanej struktury <class_name>/train/test")

    return data_dir

# 3. Preprocessing: Samplowanie punktów z plików .off i zapis do HDF5
def preprocess_modelnet10(num_points=1024):
    """Przetwarza pliki .off do chmur punktów i zapisuje do HDF5."""
    h5_path = '/content/data/modelnet10.h5'
    if os.path.exists(h5_path):
        print("Preprocessed dane już istnieją w", h5_path)
        return h5_path

    data_dir = download_modelnet10()
    os.makedirs('/content/data', exist_ok=True)

    X_train, y_train = [], []
    X_test, y_test = [], []

    for split in ['train', 'test']:
        X_split, y_split = (X_train, y_train) if split == 'train' else (X_test, y_test)
        print(f"Przetwarzanie {split} set...")

        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name, split)  # np. ModelNet10/bathtub/train/
            if not os.path.exists(class_dir):
                print(f"Uwaga: Katalog {class_dir} nie istnieje, pomijam klasę {class_name} w {split}")
                continue

            off_files = [f for f in os.listdir(class_dir) if f.endswith('.off')]
            print(f"  Klasa {class_name} ({split}): {len(off_files)} plików .off")

            for file in off_files:
                file_path = os.path.join(class_dir, file)
                try:
                    mesh = trimesh.load(file_path)
                    points, _ = trimesh.sample.sample_surface(mesh, num_points)
                    points = points - np.mean(points, axis=0)  # Centrowanie
                    points /= np.max(np.linalg.norm(points, axis=1)) + 1e-6  # Skalowanie
                    X_split.append(points)
                    y_split.append(class_to_idx[class_name])
                except Exception as e:
                    print(f"    Błąd przy pliku {file_path}: {e}")

        if split == 'train':
            X_train = np.array(X_train)
            y_train = np.array(y_split)
        else:
            X_test = np.array(X_split)
            y_test = np.array(y_split)

    # Zapisz do HDF5
    with h5py.File(h5_path, 'w') as hdf:
        hdf.create_dataset('X_train', data=X_train)
        hdf.create_dataset('y_train', data=y_train)
        hdf.create_dataset('X_test', data=X_test)
        hdf.create_dataset('y_test', data=y_test)

    print(f"Preprocessing zakończony. Kształty: Train {X_train.shape}, Test {X_test.shape}")
    return h5_path

# 4. Przygotowanie danych
class ModelNet10_Dataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.FloatTensor(data)  # (N_samples, N_points, 3)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

def load_data():
    """Ładuje dane z HDF5 i tworzy DataLoader'y."""
    h5_path = preprocess_modelnet10()

    with h5py.File(h5_path, 'r') as hdf:
        X_train = hdf['X_train'][:]
        y_train = hdf['y_train'][:]
        X_test = hdf['X_test'][:]
        y_test = hdf['y_test'][:]

    val_size = 0.2
    num_train = len(X_train)
    indices = list(range(num_train))
    np.random.shuffle(indices)
    split = int(np.floor(val_size * num_train))
    train_idx, val_idx = indices[split:], indices[:split]

    trainset = ModelNet10_Dataset(X_train[train_idx], y_train[train_idx])
    valset = ModelNet10_Dataset(X_train[val_idx], y_train[val_idx])
    testset = ModelNet10_Dataset(X_test, y_test)

    trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
    valloader = DataLoader(valset, batch_size=32, shuffle=False)
    testloader = DataLoader(testset, batch_size=32, shuffle=False)

    print(f"Dane załadowane: Train {len(trainset)}, Val {len(valset)}, Test {len(testset)}")
    return trainloader, valloader, testloader

# 5. Definicja modelu PointNet
class TNet(nn.Module):
    def __init__(self, k=3):
        super(TNet, self).__init__()
        self.k = k
        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k * k)
        self.dropout = nn.Dropout(p=0.2)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)

    def forward(self, x):
        batchsize = x.size()[0]
        x = nn.functional.relu(self.bn1(self.conv1(x)))
        x = nn.functional.relu(self.bn2(self.conv2(x)))
        x = nn.functional.relu(self.bn3(self.conv3(x)))
        x = torch.max(x, 2, keepdim=True)[0].view(batchsize, -1)
        x = nn.functional.relu(self.bn4(self.fc1(x)))
        x = nn.functional.relu(self.bn5(self.dropout(self.fc2(x))))
        x = self.fc3(x)

        # Dodaj tożsamościową macierz
        iden = torch.eye(self.k, requires_grad=True).repeat(batchsize, 1, 1).to(x.device)
        x = x.view(-1, self.k, self.k) + iden
        x = nn.functional.softmax(x, dim=1)
        return x

class PointNet(nn.Module):
    def __init__(self, num_classes=10, num_points=1024):
        super(PointNet, self).__init__()
        self.num_classes = num_classes
        self.num_points = num_points
        self.tnet1 = TNet(k=3)
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.tnet2 = TNet(k=64)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(p=0.3)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = x.transpose(1, 2)  # (batch, num_points, 3) -> (batch, 3, num_points)
        trans1 = self.tnet1(x)
        x = torch.bmm(x.transpose(2, 1), trans1).transpose(2, 1)
        x = nn.functional.relu(self.bn1(self.conv1(x)))
        trans2 = self.tnet2(x)
        x = torch.bmm(x.transpose(2, 1), trans2).transpose(2, 1)
        x = nn.functional.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = torch.max(x, 2, keepdim=True)[0].view(-1, 1024)
        x = nn.functional.relu(self.fc1(x))
        x = self.dropout(x)
        x = nn.functional.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return self.logsoftmax(x)

# 6. Funkcja trenująca
def train_model(model, trainloader, valloader, criterion, optimizer, num_epochs, patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_losses, val_losses, val_accuracies = [], [], []
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model = None

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for points, labels in trainloader:
            points, labels = points.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(points)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * points.size(0)
        train_loss /= len(trainloader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for points, labels in valloader:
                points, labels = points.to(device), labels.to(device)
                outputs = model(points)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * points.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        val_loss /= len(valloader.dataset)
        val_accuracy = 100 * correct / total

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}, Val Accuracy: {val_accuracy:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict().copy()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping after {epoch+1} epochs")
                model.load_state_dict(best_model)
                break

    return train_losses, val_losses, val_accuracies

# 7. Funkcje wizualizacyjne
def plot_metrics(train_losses, val_losses, val_accuracies):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_accuracies, label='Val Accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.tight_layout()
    plt.show()

def visualize_predictions(model, testloader, num_samples=5):
    device = next(model.parameters()).device
    model.eval()
    points, labels = next(iter(testloader))
    points, labels = points[:num_samples].to(device), labels[:num_samples].to(device)

    with torch.no_grad():
        outputs = model(points)
        _, predicted = torch.max(outputs, 1)

    points = points.cpu().numpy()

    fig = plt.figure(figsize=(15, 3))
    for i in range(num_samples):
        ax = fig.add_subplot(1, num_samples, i+1, projection='3d')
        ax.scatter(points[i, :, 0], points[i, :, 1], points[i, :, 2], s=1)
        ax.set_title(f'Pred: {classes[predicted[i].item()]}\nTrue: {classes[labels[i].item()]}')
        ax.axis('off')
    plt.show()

# 8. Walidacja krzyżowa
def cross_validation(k_folds=3, num_epochs=5):
    with h5py.File('/content/data/modelnet10.h5', 'r') as hdf:
        full_data = hdf['X_train'][:]
        full_labels = hdf['y_train'][:]

    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    results = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(full_data)):
        print(f"Fold {fold+1}/{k_folds}")
        trainset_fold = ModelNet10_Dataset(full_data[train_idx], full_labels[train_idx])
        valset_fold = ModelNet10_Dataset(full_data[val_idx], full_labels[val_idx])
        trainloader_fold = DataLoader(trainset_fold, batch_size=32, shuffle=True)
        valloader_fold = DataLoader(valset_fold, batch_size=32, shuffle=False)

        model_fold = PointNet(num_classes=10).to(device)
        criterion_fold = nn.CrossEntropyLoss()
        optimizer_fold = optim.Adam(model_fold.parameters(), lr=0.001)

        _, _, val_accuracies_fold = train_model(model_fold, trainloader_fold, valloader_fold, criterion_fold, optimizer_fold, num_epochs)
        results.append(max(val_accuracies_fold))

    print(f'Average Validation Accuracy: {np.mean(results):.2f}% ± {np.std(results):.2f}%')

# 9. Główny kod
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Ładuj dane
    try:
        trainloader, valloader, testloader = load_data()
    except Exception as e:
        print(f"Błąd podczas ładowania danych: {e}")
        raise

    # Model
    model = PointNet(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Trening
    train_losses, val_losses, val_accuracies = train_model(model, trainloader, valloader, criterion, optimizer, num_epochs=20, patience=5)

    # Wizualizacja
    plot_metrics(train_losses, val_losses, val_accuracies)
    visualize_predictions(model, testloader)

    # Walidacja krzyżowa
    cross_validation(k_folds=3, num_epochs=5)

    # Ewaluacja testowa
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for points, labels in testloader:
            points, labels = points.to(device), labels.to(device)
            outputs = model(points)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
import os
import h5py
import numpy as np
import trimesh
import kagglehub
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


classes = ['bathtub','bed','chair','desk','dresser','monitor',
           'night_stand','sofa','table','toilet']
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}


def download_modelnet10():
    path = kagglehub.dataset_download("balraj98/modelnet10-princeton-3d-object-dataset")
    data_dir = os.path.join(path, "ModelNet10")
    if not os.path.exists(data_dir):
        data_dir = path
    print("Dataset root:", data_dir)
    return data_dir


def preprocess_modelnet10(num_points=1024):
    h5_path = "/content/modelnet10.h5"
    if os.path.exists(h5_path):
        print("HDF5 already exists:", h5_path)
        return h5_path

    data_dir = download_modelnet10()

    X_train, y_train = [], []
    X_test, y_test = [], []

    for split in ["train", "test"]:
        X_out = X_train if split == "train" else X_test
        y_out = y_train if split == "train" else y_test

        print(f"\nProcessing {split}...")
        for cls in classes:
            class_dir = os.path.join(data_dir, cls, split)
            off_files = [f for f in os.listdir(class_dir) if f.endswith(".off")]
            print(f"  {cls}: {len(off_files)} files")

            for off in off_files:
                path = os.path.join(class_dir, off)
                try:
                    mesh = trimesh.load(path)
                    pts, _ = trimesh.sample.sample_surface(mesh, num_points)
                    pts = pts - pts.mean(axis=0)
                    pts /= np.max(np.linalg.norm(pts, axis=1)) + 1e-6
                    X_out.append(pts)
                    y_out.append(class_to_idx[cls])
                except Exception as e:
                    print("Error:", path, e)

    X_train, y_train = np.array(X_train), np.array(y_train)
    X_test, y_test   = np.array(X_test),  np.array(y_test)

    with h5py.File(h5_path, "w") as f:
        f.create_dataset("X_train", data=X_train)
        f.create_dataset("y_train", data=y_train)
        f.create_dataset("X_test",  data=X_test)
        f.create_dataset("y_test",  data=y_test)

    print("\nSaved HDF5:", h5_path)
    return h5_path



class ModelNet10Dataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.FloatTensor(data)
        self.labels = torch.LongTensor(labels)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]


def load_data():
    h5_path = preprocess_modelnet10()

    with h5py.File(h5_path, "r") as f:
        X_train, y_train = f["X_train"][:], f["y_train"][:]
        X_test,  y_test  = f["X_test"][:],  f["y_test"][:]

    N = len(X_train)
    idx = np.random.permutation(N)
    split = int(0.2*N)

    train = ModelNet10Dataset(X_train[idx[split:]], y_train[idx[split:]])
    val   = ModelNet10Dataset(X_train[idx[:split]],  y_train[idx[:split]])
    test  = ModelNet10Dataset(X_test, y_test)

    train_loader = DataLoader(train, batch_size=32, shuffle=True)
    val_loader   = DataLoader(val,   batch_size=32)
    test_loader  = DataLoader(test,  batch_size=32)

    print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")
    return train_loader, val_loader, test_loader


# ============================================
# PointNet (oryginalna architektura)
# ============================================
class TNet(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.k = k
        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64,128,1)
        self.conv3 = nn.Conv1d(128,1024,1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.fc1 = nn.Linear(1024,512)
        self.fc2 = nn.Linear(512,256)
        self.fc3 = nn.Linear(256,k*k)

    def forward(self,x):
        B = x.size(0)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.relu(self.bn3(self.conv3(x)))
        x = torch.max(x,2)[0]
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        I = torch.eye(self.k).to(x.device).view(1,self.k,self.k)
        return x.view(-1,self.k,self.k) + I


class PointNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.t1 = TNet(3)
        self.conv1 = nn.Conv1d(3,64,1)
        self.bn1 = nn.BatchNorm1d(64)

        self.t2 = TNet(64)
        self.conv2 = nn.Conv1d(64,128,1)
        self.conv3 = nn.Conv1d(128,1024,1)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)

        self.fc1 = nn.Linear(1024,512)
        self.fc2 = nn.Linear(512,256)
        self.fc3 = nn.Linear(256,num_classes)

    def forward(self,x):
        x = x.transpose(1,2)
        t = self.t1(x)
        x = torch.bmm(x.transpose(2,1), t).transpose(2,1)
        x = torch.relu(self.bn1(self.conv1(x)))

        t2 = self.t2(x)
        x = torch.bmm(x.transpose(2,1), t2).transpose(2,1)

        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = torch.max(x,2)[0]

        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# ============================================
# Model Fine-Tuning (Transfer Learning)
# ============================================
class PointNet_FT(nn.Module):
    def __init__(self, pretrained, num_classes=10):
        super().__init__()

        self.backbone = pretrained

        # Nadpisujemy classifier
        self.backbone.fc1 = nn.Linear(1024, 512)
        self.backbone.fc2 = nn.Linear(512, 256)
        self.backbone.fc3 = nn.Linear(256, num_classes)

        # Freeze all except top layers
        for name, param in self.backbone.named_parameters():
            param.requires_grad = ("conv3" in name or "fc" in name)

    def forward(self, x):
        return self.backbone(x)


# ============================================
# 8. Trening Fine-Tuning
# ============================================
def train_ft(model, train_loader, val_loader, epochs=15, lr=1e-4, patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optim_ft = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )
    criterion = nn.CrossEntropyLoss()

    best_loss = float("inf")
    best_weights = None
    no_imp = 0
    acc_hist = []

    for ep in range(epochs):
        model.train()
        for pts, lbl in train_loader:
            pts, lbl = pts.to(device), lbl.to(device)
            optim_ft.zero_grad()
            out = model(pts)
            loss = criterion(out, lbl)
            loss.backward()
            optim_ft.step()

        # --- validation ---
        model.eval()
        correct, total, vloss = 0, 0, 0
        with torch.no_grad():
            for pts, lbl in val_loader:
                pts, lbl = pts.to(device), lbl.to(device)
                out = model(pts)
                loss = criterion(out, lbl)
                vloss += loss.item()
                _, pred = torch.max(out,1)
                correct += (pred == lbl).sum().item()
                total += lbl.size(0)

        vloss /= len(val_loader)
        vacc = 100 * correct / total
        acc_hist.append(vacc)

        print(f"[FT] Epoch {ep+1}/{epochs} | Val Loss {vloss:.4f} | Val Acc {vacc:.2f}%")

        if vloss < best_loss:
            best_loss = vloss
            best_weights = model.state_dict().copy()
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print("Early stopping!")
                break

    model.load_state_dict(best_weights)
    return model, acc_hist


# ============================================
# 9. MAIN — Fine-Tuning
# ============================================
if __name__ == "__main__":
    train_loader, val_loader, test_loader = load_data()

    # --- 1) Trenujemy mały model bazowy jako "pretrained"
    pretrained = PointNet(num_classes=10)
    pretrained, _ = train_ft(pretrained, train_loader, val_loader, epochs=3, lr=0.001)

    # zapisujemy checkpoint
    torch.save(pretrained.state_dict(), "/content/pretrained_pointnet.pth")

    # --- 2) Wczytujemy pretrained i fine-tunujemy
    base = PointNet(num_classes=10)
    base.load_state_dict(torch.load("/content/pretrained_pointnet.pth"))

    ft_model = PointNet_FT(base, num_classes=10)

    ft_model, ft_acc = train_ft(ft_model, train_loader, val_loader, epochs=15, lr=1e-4)

    # --- 3) Test accuracy
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ft_model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for pts, lbl in test_loader:
            pts, lbl = pts.to(device), lbl.to(device)
            out = ft_model(pts)
            _, pred = torch.max(out, 1)
            correct += (pred == lbl).sum().item()
            total += lbl.size(0)

    print(f"\nFinal Test Accuracy (Fine-Tuning): {100*correct/total:.2f}%")

    # --- 4) Wykres
    plt.plot(ft_acc)
    plt.title("Validation Accuracy — Fine-Tuning PointNet")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy [%]")
    plt.show()


100%|██████████| 454M/454M [00:03<00:00, 144MB/s]

Extracting files...


Dataset root: /root/.cache/kagglehub/datasets/balraj98/modelnet10-princeton-3d-object-dataset/versions/1/ModelNet10

Processing train...
  bathtub: 106 files
  bed: 515 files
  chair: 889 files
  desk: 200 files
  dresser: 200 files
  monitor: 465 files
  night_stand: 200 files
  sofa: 680 files
  table: 392 files
  toilet: 344 files

Processing test...
  bathtub: 50 files
  bed: 100 files
  chair: 100 files
  desk: 86 files
  dresser: 86 files
  monitor: 100 files
  night_stand: 86 files
  sofa: 100 files
  table: 100 files
  toilet: 100 files

Saved HDF5: /content/modelnet10.h5
Train: 3193, Val: 798, Test: 908
